## Preprocessing Data

In [123]:
import pandas as pd 

df = pd.read_csv("../data.csv")
print("Original dataset shape:", df.shape)
df.head()

Original dataset shape: (974, 3)


,Temp,Humd,Label
0,23.0,17,1
1,23.0,17,1
2,23.0,17,1
3,23.0,17,1
4,23.0,17,1


Adding False values:


In [124]:
import pandas as pd # in case of error, install pnadas using: pip install pandas
import random

random.seed(42) #makes sure that the random numbers generated are the same every time you run the code.
 
# Read the CSV file into a DataFrame

df = pd.read_csv("../data.csv")

false_rows = []

for i in range(100):
    
    false_rows.append({       #using random integers to generate false data.
        "Temp": random.randint(-100, 200),
        "Humd": random.randint(0, 100), 
        "Label": 0
    })
    
df_new = pd.DataFrame(false_rows)
df = pd.concat([df, df_new], ignore_index=True)

# Step 3: Save the DataFrame to a new CSV file
df.to_csv("data_hacked.csv", index=False)

print("New dataset shape:", df.shape)
df.head()
df.tail()

New dataset shape: (1074, 3)


,Temp,Humd,Label
1069,172.0,98,0
1070,-36.0,16,0
1071,143.0,70,0
1072,-16.0,33,0
1073,170.0,77,0


Data Pre processing: 

In [125]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values)

#remove duplicates
df = df.drop_duplicates()
print("processed dataset shape:", df.shape)

Missing Values:
 Temp     0
Humd     0
Label    0
dtype: int64
processed dataset shape: (147, 3)


Feature prep


In [126]:
# prepare features
x = df.drop(['Label'],axis=1) #remove class or label
y = df['Label'] #load label

data split:

## Algorithm 1: Linear Classification

In [127]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler #module for perform scaling

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

#metrics for evaluating model performance
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, classification_report

models = {              #many models to loop thru
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "SVM (RBF)": SVC(kernel="rbf"),
    "KNN (5 neighbors)": KNeighborsClassifier(n_neighbors=5),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
}

random_seeds = [10, 22, 42, 99, 123, 70, 128, 256, 512, 1024] #list of random seeds to loop thru


results = [] #stores the model results

    
#25-75 split for training and testing data, good for small models
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25,  random_state=42)

# Scale the data using standardization
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train) #scale training set
x_test = scaler.transform(x_test) #scale test set




for name, model in models.items():
    train_scores = []
    test_scores = []
    
    for seed in random_seeds:
        # Split the data into training and testing sets with the current random seed
        x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=seed)
        
        # Scale the data using standardization
        scaler = StandardScaler()
        x_train = scaler.fit_transform(x_train)  # scale training set
        x_test = scaler.transform(x_test)  # scale test set
    
        model.fit(x_train, y_train)

        train_predictions = model.predict(x_train)
        test_predictions = model.predict(x_test)
        
        train_acc = accuracy_score(y_train, train_predictions)
        test_acc = accuracy_score(y_test, test_predictions)
        
        #appends to temp list
        train_scores.append(train_acc)
        test_scores.append(test_acc)
    
    avg_train_acc = np.mean(train_scores)
    avg_test_acc = np.mean(test_scores)
    
    '''
    print(f"\n--- {name} ---")
    print(f"Training accuracy: {train_acc:.3f}")
    print(f"Testing accuracy: {test_acc:.3f}")
    '''
    print(f"\n{name} done training and testing.")
    #results.append((name, test_acc, train_acc))
    
    results.append({
        "Model Name": name,
        "Avg Training Accuracy": round(avg_train_acc, 3),
        "Avg Testing Accuracy": round(avg_test_acc, 3)
    })

    
results_df = pd.DataFrame(results)

results_df = results_df.sort_values(by="Avg Testing Accuracy", ascending=False).reset_index(drop=True)



#comparison table
print("\n--- Final Model Performance Comparison ---")
results_df


Logistic Regression done training and testing.

Random Forest done training and testing.

SVM (RBF) done training and testing.

KNN (5 neighbors) done training and testing.

Decision Tree done training and testing.

--- Final Model Performance Comparison ---


,Model Name,Avg Training Accuracy,Avg Testing Accuracy
0,Random Forest,1.000,0.983
1,Decision Tree,1.000,0.963
2,KNN (5 neighbors),0.932,0.940
3,SVM (RBF),0.922,0.937
4,Logistic Regression,0.606,0.553


Saves best model:

In [128]:
best_model_name = results_df.loc[0, "Model Name"]
print(f"The best performing model was: {best_model_name}")

best_model_object = models[best_model_name]


# library for save and load scikit-learn models
import pickle
# file name, recommending *.pickle as a file extension
filename = "best_model.pickle"

# save model
pickle.dump(best_model_object, open(filename, "wb"))
print(f"Successfully saved {best_model_name} to {filename}!")


The best performing model was: Random Forest
Successfully saved Random Forest to best_model.pickle!
